In [0]:
spark.sql("""
CREATE OR REPLACE TABLE bank_accounts_cdf_new (
  account_id INT,
  balance INT
) USING DELTA
""")

DataFrame[]

In [0]:
spark.sql("""
ALTER TABLE bank_accounts_cdf_new
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)
""")

DataFrame[]

In [0]:
spark.sql("""
INSERT INTO bank_accounts_cdf_new VALUES
(1001, 1000),
(1002, 2000)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
UPDATE bank_accounts_cdf_new
SET balance = 2500
WHERE account_id = 1002
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("""
INSERT INTO bank_accounts_cdf_new VALUES
(1003, 3000)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
DELETE FROM bank_accounts_cdf_new
WHERE account_id = 1001
""")

DataFrame[num_affected_rows: bigint]

In [0]:
%sql

DESCRIBE HISTORY databricks_carltest.default.bank_accounts_cdf_new;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-02-16T01:23:40Z,140872976473246,carlcheno@hotmail.com,DELETE,"Map(predicate -> [""(account_id#8327 = 1001)""])",null,List(535108728348707),0207-221932-b7ozbf6g,6,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 906, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1047, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 783, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 264)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
6,2026-02-16T01:23:36Z,140872976473246,carlcheno@hotmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(535108728348707),0207-221932-b7ozbf6g,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 897)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
5,2026-02-16T01:23:26Z,140872976473246,carlcheno@hotmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(535108728348707),0207-221932-b7ozbf6g,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2250, p25FileSize -> 906, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 217, minFileSize -> 906, numAddedFiles -> 1, maxFileSize -> 906, p75FileSize -> 906, p50FileSize -> 906, numAddedBytes -> 906)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
4,2026-02-16T01:23:25Z,140872976473246,carlcheno@hotmail.com,UPDATE,"Map(predicate -> [""(account_id#7339 = 1002)""])",null,List(535108728348707),0207-221932-b7ozbf6g,3,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1344, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 3163, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2000, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1344, rewriteTimeMs -> 1162)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
3,2026-02-16T01:23:21Z,140872976473246,carlcheno@hotmail.com,UPDATE,"Map(predicate -> [""(account_id#6527 = 1002)""])",null,List(535108728348707),0207-221932-b7ozbf6g,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 2964, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1212, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1344, rewriteTimeMs -> 1736)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
2,2026-02-16T01:23:14Z,140872976473246,carlcheno@hotmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(535108728348707),0207-221932-b7ozbf6g,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 906)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
1,2026-02-16T01:23:02Z,140872976473246,carlcheno@hotmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(535108728348707),0207-221932-b7ozbf6g,0,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13
0,2026-02-16T01:23:00Z,140872976473246,carlcheno@hotmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(535108728348707),0207-221932-b7ozbf6g,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.13


In [0]:
spark.read.format("delta") \
  .option("readChangeFeed", "true") \
  .option("startingVersion", 2) \
  .table("bank_accounts_cdf") \
  .show()

+----------+-------+----------------+---------------+-------------------+
|account_id|balance|    _change_type|_commit_version|  _commit_timestamp|
+----------+-------+----------------+---------------+-------------------+
|      1002|   2500| update_preimage|             10|2026-02-16 01:15:25|
|      1002|   2500|update_postimage|             10|2026-02-16 01:15:25|
|      1002|   2500| update_preimage|             10|2026-02-16 01:15:25|
|      1002|   2500|update_postimage|             10|2026-02-16 01:15:25|
|      1002|   2500| update_preimage|              8|2026-02-16 01:14:44|
|      1002|   2500|update_postimage|              8|2026-02-16 01:14:44|
|      1002|   2000| update_preimage|              4|2026-02-16 01:14:02|
|      1002|   2500|update_postimage|              4|2026-02-16 01:14:02|
|      1002|   2000| update_preimage|              8|2026-02-16 01:14:44|
|      1002|   2500|update_postimage|              8|2026-02-16 01:14:44|
|      1001|   1000|          insert| 

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'